# Test the L0 processor

* https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-790
* https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-817

In [ ]:
# Experimental DPR processor configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 4,
        "memory_limit": "58GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
cluster_info = await init_dask_cluster_l0()

In [ ]:
# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcessor

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-warning">

Note: for now the L0 processor returns dummy values that are not usable.
</div>

In [ ]:
tasktable: dict = dpr_client.get_process(DprProcessor.S1L0.value, cluster_info)
print(f"Tasktable for {DprProcessor.S1L0.value!r}:")
display(JSON(tasktable))
# print(json.dumps(tasktable, indent=2))

## Init environment for the processors

In [ ]:
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir="./config"
)

await dpr.init(local_secrets_file="./config/secrets.json")

# Arguments for S1 L0
s1_args = {
    "process": DprProcessor.S1L0.value, 
    "cluster_info": cluster_info,
    "payload_subpath": "s1/basic_payload.yaml",
    "experimental_config": experimental_config,    
}

## Run the S1 L0 processor

In [ ]:
# Run S1 short data
if os.getenv("RSPY_FROM_CICD") != "1":
    s1_output_dir = osp.join(dpr.s3_output_dir, "dask_cluster", "l0", "s1.short")
    print(f"s1_output_dir = {s1_output_dir}")
    await dpr.run_process(
        **s1_args,
        s3_output_dir = s1_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "dask_cluster", "l0", "s1.short"),
        # Payload env vars
        OUTPUT_DIR = s1_output_dir,
        SHORT_SUFFIX=".short",
    )


In [ ]:
# Run S1 full data
if os.getenv("RSPY_FROM_CICD") != "1":
    s1_output_dir = osp.join(dpr.s3_output_dir, "dask_cluster", "l0", "s1")
    await dpr.run_process(
        **s1_args,
        s3_output_dir = s1_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "dask_cluster", "l0", "s1"),
        # Payload env vars
        OUTPUT_DIR = s1_output_dir,
        SHORT_SUFFIX="",
    )